In [10]:
from firecrawl import Firecrawl

# init firecrawl
# IMPORTANT: Since you shared this key publicly, you should revoke it and generate a new one!
app = Firecrawl(api_key="fc-12df949527d445b7af607ce7325da432")

# scrape page (Notice the method is now just `scrape`)
result = app.scrape(
    "https://www.matrixcomsec.com/",
    formats=["markdown"]
)

# Extract markdown (Depending on the exact SDK version, it might return a dict or an object)
markdown_text = result["markdown"] if isinstance(result, dict) else result.markdown

# save to txt
with open("matrix_products_all.txt", "w", encoding="utf-8") as f:
    f.write(markdown_text)

print("Saved to matrix_products.txt")

Saved to matrix_products.txt


In [11]:
from firecrawl import Firecrawl
app = Firecrawl(api_key="fc-12df949527d445b7af607ce7325da432")

print("Starting deep crawl... this might take a minute.")

# Crawl the website, follow sub-pages, and strip boilerplate
crawl_result = app.crawl(
    url="https://www.matrixcomsec.com/products/",
    limit=10, # Max number of sub-pages to crawl. Increase this if you want more!
    scrape_options={
        "formats": ["markdown"],
        "onlyMainContent": True  # <--- THIS STRIPS OUT THE BANNERS AND MENUS
    }
)

# Loop through all the sub-pages it found and save them as separate files
for i, page in enumerate(crawl_result.get('data', [])):
    # Grab the URL so we know which page this is
    url = page.get('metadata', {}).get('sourceURL', f'page_{i}')
    markdown_text = page.get('markdown', '')
    
    # Create a safe file name based on the end of the URL
    safe_name = url.strip('/').split('/')[-1] if url != "https://www.matrixcomsec.com/products/" else "main_products"
    filename = f"{safe_name}.txt"
    
    # Save the deep details
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"Source URL: {url}\n\n")
        f.write(markdown_text)
        
    print(f"Saved detail page to {filename}")

Starting deep crawl... this might take a minute.


AttributeError: 'CrawlJob' object has no attribute 'get'

In [12]:
crawl_result

CrawlJob(status='completed', total=10, completed=10, credits_used=10, expires_at=datetime.datetime(2026, 5, 1, 19, 5, 24, tzinfo=TzInfo(0)), next=None, data=[Document(markdown='Rotate(Landscape) your phone for better view.\n\nVideo Surveillance\n\nTime-Attendance\n\nAccess Control\n\nTelecom\n\nVideo Surveillance\n\nNetwork Cameras\n\nNetwork Video Recorders (NVRs)\n\nVideo Management Software (VMS)\n\nNetwork Cameras\n\n|     |     |     |     |     |     |\n| --- | --- | --- | --- | --- | --- |\n| Bullet Network Camera\n\n2MP5MP8MP\n\n![](https://www.matrixcomsec.com/products/wp-content/uploads/2022/04/New_cam_img_1.png)\n\nKey Features\n\n- Sony STARVIS Series – Back Illuminated CMOS Sensor\n- Higher Field of View\n- Superlative Image with WDR\n- H.265 Compression\n- Clear Night Vision\n- 512GB SD Card Support\n- High Signal to Noise Ratio\n- Real-time Notification\n\nProject Series\n\n|     |     |     |     |     |     |\n| --- | --- | --- | --- | --- | --- |\n| Model Name | Resol

In [13]:
from firecrawl import Firecrawl

app = Firecrawl(api_key="fc-12df949527d445b7af607ce7325da432")
print("Starting deep crawl... this might take a minute.")

# Crawl the website, follow sub-pages, and strip boilerplate
crawl_result = app.crawl(
    url="https://www.matrixcomsec.com/",
    limit=10, 
    scrape_options={
        "formats": ["markdown"],
        "onlyMainContent": True
    }
)

# FIXED: crawl_result is an object, so we access the 'data' attribute directly
# We use 'or []' as a fallback just in case the crawl returns nothing
pages = crawl_result.data or []

for i, page in enumerate(pages):
    # FIXED: 'page' is also an object, so we use dot notation instead of .get()
    
    # 1. Safely extract the URL
    url = f'page_{i}'
    if page.metadata and page.metadata.source_url:
        url = page.metadata.source_url
        
    # 2. Extract the markdown text
    markdown_text = page.markdown or ''
    
    # Create a safe file name based on the end of the URL
    safe_name = url.strip('/').split('/')[-1] if url != "https://www.matrixcomsec.com/products/" else "main_products"
    
    # Fallback if safe_name ends up completely empty
    if not safe_name:
        safe_name = f"page_{i}"
        
    filename = f"{safe_name}.txt"
    
    # Save the deep details
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"Source URL: {url}\n\n")
        f.write(markdown_text)
        
    print(f"Saved detail page to {filename}")

Starting deep crawl... this might take a minute.
Saved detail page to www.matrixcomsec.com.txt
Saved detail page to telecom.txt
Saved detail page to time-attendance.txt
Saved detail page to access-control-solution.txt
Saved detail page to lost-password.txt
Saved detail page to main_products.txt
Saved detail page to sitemap.xml.txt
Saved detail page to 2mp-ir-dome-camera-with-audio-support-3-6mm-lens.txt
Saved detail page to access-control.txt
Saved detail page to shop.txt


In [21]:
import os
import re
from firecrawl import Firecrawl

# 1. Initialize Firecrawl

app = Firecrawl(api_key="fc-12df949527d445b7af607ce7325da432")

# 2. The organized data extracted from your HTML
matrix_products = {
    # "Video Surveillance": {
    #     "Project Bullet Network Cameras": "https://www.matrixcomsec.com/product/project-series-bullet-ip-cameras/",
    #     "Mini Bullet Network Cameras": "https://www.matrixcomsec.com/product/mini-bullet-ip-cameras/",
    #     "Project Dome Network Cameras": "https://www.matrixcomsec.com/product/project-series-dome-ip-cameras/",
    #     "Mini Dome Network Cameras": "https://www.matrixcomsec.com/product/mini-dome-ip-cameras/",
    #     "PTZ IR Cameras": "https://www.matrixcomsec.com/product/ptz-camera/",
    #     "Ruggedized Network Cameras": "https://www.matrixcomsec.com/product/ruggedized-network-cameras/",
    #     "Turret Network Cameras": "https://www.matrixcomsec.com/product/turret-network-cameras/",
    #     "VMS": "https://www.matrixcomsec.com/product/satatya-samas-video-management-system/",
    #     "Network Video Recorders": "https://www.matrixcomsec.com/product/satatya-network-video-recorders/"
    # }
    # "Access Control": {
    #     "Two Door Controller": "https://www.matrixcomsec.com/product/cosec-arc-dc200p-door-access-control/",
    #     "Face-based Door Controllers": "https://www.matrixcomsec.com/product/cosec-argo-face-door-controllers/",
    #     "Ergonomic Biometric Door Controllers": "https://www.matrixcomsec.com/product/cosec-argo-door-access-control/",
    #     "Aux-supported Biometric Door Controllers": "https://www.matrixcomsec.com/product/cosec-vega-door-access-control/",
    #     "Compact Biometric Door Controllers": "https://www.matrixcomsec.com/product/cosec-path-series-door-controllers/",
    #     "Compact Reader Series": "https://www.matrixcomsec.com/product/cosec-path-access-control-reader/",
    #     "Touch Screen Reader Series": "https://www.matrixcomsec.com/product/cosec-atom-access-control-reader/",
    #     "COSEC PANEL200P": "https://www.matrixcomsec.com/product/cosec-panel200p-access-control-panel/",
    #     "COSEC ARC IO800": "https://www.matrixcomsec.com/product/cosec-arc-io800-input-output-controller/",
    #     "Cloud-based": "https://www.matrixcomsec.com/product/cosec-vyom/",
    #     "Server-based": "https://www.matrixcomsec.com/product/cosec-centra/",
    #     "Face Recognition Solution": "https://www.matrixcomsec.com/?post_type=product&p=15230",
    #     "Accessories": "https://www.matrixcomsec.com/product/cosec-access-control-accessories/"
    # }
    # "Time Attendance": {
    #     "Face-based Door Controllers": "https://www.matrixcomsec.com/product/cosec-argo-face-door-controllers/",
    #     "Ergonomic Biometric Door Controllers": "https://www.matrixcomsec.com/product/cosec-argo-door-access-control/",
    #     "Aux-supported Biometric Door Controllers": "https://www.matrixcomsec.com/product/cosec-vega-door-access-control/",
    #     "Compact Biometric Door Controllers": "https://www.matrixcomsec.com/product/cosec-path-series-door-controllers/",
    #     "Cloud-based": "https://www.matrixcomsec.com/product/cosec-vyom/",
    #     "Server-based": "https://www.matrixcomsec.com/product/cosec-centra/",
    #     "Mobile-based Employee Portal": "https://www.matrixcomsec.com/product/cosec-apta-mobile-app/",
    #     "Time-Attendance Solution for SOHO": "https://www.matrixcomsec.com/product/cosec-samay-time-attendance-solution/",
    #     "Face Recognition Solution": "https://www.matrixcomsec.com/?post_type=product&p=15230",
    #     "Accessories": "https://www.matrixcomsec.com/product/cosec-access-control-accessories/"
    # },
    "Telecom": {
        "Small Modular PBX VISIONPRO": "https://www.matrixcomsec.com/product/visionpro-soho-pbx-system/",
        "Small Modular Hybrid IP PBX ETERNITY NENX": "https://www.matrixcomsec.com/product/eternity-nenx-ip-pbx-system/",
        "Embedded IP-PBX Server PRASAR UCS SPARK200": "https://www.matrixcomsec.com/product/prasar-ip-pbx-server/",
        "IP-PBX Server Application ANANT": "https://www.matrixcomsec.com/product/anant-ip-pbx-server-application/",
        "Hybrid IP PBX for Small Business ETERNITY PENX": "https://www.matrixcomsec.com/product/eternity-penx-hybrid-ip-pbx/",
        "Hybrid IP-PBX for SME ETERNITY GENX": "https://www.matrixcomsec.com/product/eternity-genx-ip-pbx-system/",
        "Hybrid IP-PBX for Large Enterprise ETERNITY MENX LENX": "https://www.matrixcomsec.com/product/eternity-hybrid-ip-pbx-system/",
        "GSM FCT Gateway SIMADO GFX11": "https://www.matrixcomsec.com/product/simado-gfx11-gsm-fct-gateway/",
        "SETU VFXTH VOIP FXO FXS Gateways": "https://www.matrixcomsec.com/product/setu-vfxth-voip-fxo-fxs-gateway/",
        "SETU VFX VOIP FXO FXS Gateway": "https://www.matrixcomsec.com/product/setu-vfx-voip-fxo-fxs-gateway/",
        "VOIP-PRI Gateway SETU VTEP": "https://www.matrixcomsec.com/product/setu-vtep-voip-pri-gateway/",
        "GSM FXS Gateway SIMADO GFX44": "https://www.matrixcomsec.com/product/simado-gfx44-gsm-fxs-gateway/",
        "Universal Media Gateway Application SARVAM UMG": "https://www.matrixcomsec.com/product/sarvam-universal-media-gateway/",
        "Entry Level IP Phone SPARSH VP210": "https://www.matrixcomsec.com/product/sparsh-vp210-voice-ip-phone/",
        "Premium IP Phone SPARSH VP510E": "https://www.matrixcomsec.com/product/sparsh-vp510e-ip-phone/",
        "Digital Key Phone EON510": "https://www.matrixcomsec.com/product/eon510-phone/",
        "SARVAM UCS UC Server": "https://www.matrixcomsec.com/product/sarvam-unified-communication-server/",
        "VARTA Softphone": "https://www.matrixcomsec.com/product/varta-softphone/"
    }
}

# 3. Helper function to clean text so it's safe to use as a file name
def clean_filename(name):
    # Removes invalid characters like / \ : * ? " < > |
    return re.sub(r'[\\/*?:"<>|]', "", name).strip()

print("Setting up folders and starting extraction...\n")

# 4. Loop through the categories and products
for category, products in matrix_products.items():
    # Create the directory structure: new/Category_Name
    folder_path = os.path.join("new", clean_filename(category))
    os.makedirs(folder_path, exist_ok=True)
    
    print(f"📁 Processing Category: {category} (Folder created)")
    
    for product_name, url in products.items():
        print(f"  └── Scraping: {product_name}...")
        
        try:
            # Scrape the page
            result = app.scrape(
                url=url,
                formats=["markdown"],
                only_main_content=True
            )
            
            # Safely extract markdown
            if isinstance(result, dict):
                markdown_text = result.get('markdown', '')
            else:
                markdown_text = getattr(result, 'markdown', '')
            
            # Generate the file path: new/Category_Name/Product_Name.txt
            safe_file_name = clean_filename(product_name) + ".txt"
            file_path = os.path.join(folder_path, safe_file_name)
            
            # Write data to the text file
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(f"Product Name: {product_name}\n")
                f.write(f"Source URL: {url}\n")
                f.write("====================================================\n\n")
                f.write(markdown_text)
                
            print(f"      ✅ Saved: {file_path}")
            
        except Exception as e:
            print(f"      ❌ Failed to scrape {product_name}. Error: {e}")

print("\n🎉 All done! Everything is organized in the 'new' folder.")

Setting up folders and starting extraction...

📁 Processing Category: Telecom (Folder created)
  └── Scraping: Small Modular PBX VISIONPRO...
      ✅ Saved: new/Telecom/Small Modular PBX VISIONPRO.txt
  └── Scraping: Small Modular Hybrid IP PBX ETERNITY NENX...
      ✅ Saved: new/Telecom/Small Modular Hybrid IP PBX ETERNITY NENX.txt
  └── Scraping: Embedded IP-PBX Server PRASAR UCS SPARK200...
      ✅ Saved: new/Telecom/Embedded IP-PBX Server PRASAR UCS SPARK200.txt
  └── Scraping: IP-PBX Server Application ANANT...
      ✅ Saved: new/Telecom/IP-PBX Server Application ANANT.txt
  └── Scraping: Hybrid IP PBX for Small Business ETERNITY PENX...
      ✅ Saved: new/Telecom/Hybrid IP PBX for Small Business ETERNITY PENX.txt
  └── Scraping: Hybrid IP-PBX for SME ETERNITY GENX...
      ✅ Saved: new/Telecom/Hybrid IP-PBX for SME ETERNITY GENX.txt
  └── Scraping: Hybrid IP-PBX for Large Enterprise ETERNITY MENX LENX...
      ✅ Saved: new/Telecom/Hybrid IP-PBX for Large Enterprise ETERNITY MENX 

In [22]:
import os
import re

# The folder where all your extracted data is currently stored
base_folder = "from_website"

# A regex pattern that looks for any standard web URL
url_pattern = re.compile(r'https?://')

print("Starting cleanup process...\n")

files_cleaned = 0
lines_removed_total = 0

# 1. Walk through all folders and sub-folders inside 'new/'
for root, dirs, files in os.walk(base_folder):
    for file in files:
        if file.endswith(".txt"):
            file_path = os.path.join(root, file)
            
            # 2. Read all the lines currently in the file
            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
                
            cleaned_lines = []
            lines_removed_in_file = 0
            
            # 3. Check every line one by one
            for line in lines:
                # If the line has NO url OR if it's our official tracking header... keep it!
                if not url_pattern.search(line) or line.startswith("Source URL:"):
                    cleaned_lines.append(line)
                else:
                    # If it has a URL (and isn't the header), we drop it
                    lines_removed_in_file += 1
            
            # 4. Overwrite the file with only the clean lines
            if lines_removed_in_file > 0:
                with open(file_path, "w", encoding="utf-8") as f:
                    f.writelines(cleaned_lines)
                
                print(f"🧹 Cleaned: {file} (Removed {lines_removed_in_file} lines with URLs)")
                files_cleaned += 1
                lines_removed_total += lines_removed_in_file

print(f"\n✅ Cleanup Complete! Removed a total of {lines_removed_total} useless link/image lines across {files_cleaned} files.")

Starting cleanup process...

🧹 Cleaned: Mini Dome Network Cameras.txt (Removed 277 lines with URLs)
🧹 Cleaned: VMS.txt (Removed 228 lines with URLs)
🧹 Cleaned: Network Video Recorders.txt (Removed 276 lines with URLs)
🧹 Cleaned: PTZ IR Cameras.txt (Removed 244 lines with URLs)
🧹 Cleaned: Project Bullet Network Cameras.txt (Removed 287 lines with URLs)
🧹 Cleaned: Turret Network Cameras.txt (Removed 292 lines with URLs)
🧹 Cleaned: Mini Bullet Network Cameras.txt (Removed 278 lines with URLs)
🧹 Cleaned: Ruggedized Network Cameras.txt (Removed 289 lines with URLs)
🧹 Cleaned: Project Dome Network Cameras.txt (Removed 281 lines with URLs)
🧹 Cleaned: Touch Screen Reader Series.txt (Removed 251 lines with URLs)
🧹 Cleaned: COSEC PANEL200P.txt (Removed 215 lines with URLs)
🧹 Cleaned: Face Recognition Solution.txt (Removed 495 lines with URLs)
🧹 Cleaned: Two Door Controller.txt (Removed 215 lines with URLs)
🧹 Cleaned: Compact Reader Series.txt (Removed 230 lines with URLs)
🧹 Cleaned: Accessories.